# 02. 논문 수식을 코드로 검증하기

이 노트북은 논문에 쓸 수식 (1)~(5)가 실제 모델과 코드에서 성립하는지를 셀 단위로 확인합니다. 각 수식은 [마크다운: 수식과 설명] → [코드: 구현] → [코드: 수치 검증 또는 플롯] 세 셀로 묶여 있습니다. 아래에 적힌 수치는 모두 셀이 실제로 출력한 값이며, 실행 전에는 비어 있습니다.

`MODE`는 두 가지입니다.

| MODE | 모델 | 용도 |
|---|---|---|
| `tiny` (기본) | 랜덤 초기화 소형 Qwen3_5Config (tests/test_smoke_cpu.py와 같은 설정), CPU | 수식과 코드의 일치 여부만 봅니다. 모델 통계는 의미가 없습니다 |
| `real` | Qwen/Qwen3.5-0.8B, Colab GPU (T4면 fp32) | 논문에 들어갈 수치와 그림을 만듭니다 |

환경변수 `HQ_NB_MODE`로도 바꿀 수 있습니다.

In [ ]:
import os, sys, math, json, glob, time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

MODE = os.environ.get("HQ_NB_MODE", "tiny")       # "tiny" 또는 "real"
REAL_MODEL = "Qwen/Qwen3.5-0.8B"

# 레포 루트를 찾습니다 (노트북은 notebooks/ 에서 실행됩니다)
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from hybridquant import (HybridQuantConfig, QuantSpec, QuantStats, fake_quant, quantized_cache,
                         make_cache, decode_nll, load_model)
from transformers import Qwen3_5TextConfig, Qwen3_5ForCausalLM

torch.manual_seed(0); np.random.seed(0)


def tiny_config():
    # tests/test_smoke_cpu.py 의 _tiny_config 와 같은 값입니다
    return Qwen3_5TextConfig(
        vocab_size=256, hidden_size=64, intermediate_size=128, num_hidden_layers=4,
        num_attention_heads=2, num_key_value_heads=1, head_dim=32,
        linear_num_key_heads=2, linear_num_value_heads=2, linear_key_head_dim=16, linear_value_head_dim=16,
        linear_conv_kernel_dim=4, max_position_embeddings=512,
        layer_types=["linear_attention", "linear_attention", "linear_attention", "full_attention"])


def load():
    if MODE == "tiny":
        return Qwen3_5ForCausalLM(tiny_config()).eval(), None
    dtype = "auto"
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
        dtype = "fp32"            # T4: bf16 텐서코어가 없고, 수식 검증에는 fp32 가 안전합니다
    return load_model(REAL_MODEL, dtype=dtype)


model, tok = load()
DEVICE = next(model.parameters()).device
CFG = model.config.get_text_config(decoder=True)
PREFILL, DECODE = (16, 8) if MODE == "tiny" else (256, 128)


def sample_ids(n):
    """tiny 모드는 랜덤 토큰, real 모드는 짧은 영어 문장을 반복한 토큰입니다."""
    if tok is None:
        g = torch.Generator().manual_seed(1234)
        return torch.randint(0, CFG.vocab_size, (1, n), generator=g)
    text = "The theory of general relativity describes gravity as a geometric property of spacetime. " * (n // 8 + 1)
    return tok(text, return_tensors="pt")["input_ids"][:, :n]


plt.rcParams.update({"figure.figsize": (6, 3.6), "figure.dpi": 110, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3, "savefig.bbox": "tight"})
FIG_DIR = ROOT / "paper" / "figures" / ("debug" if MODE == "tiny" else "")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(name):
    p = FIG_DIR / f"{name}.pdf"
    plt.savefig(p)
    print("saved", p.relative_to(ROOT))
    return p


RESULTS = {}   # 마지막 "검증 요약" 셀에서 표로 출력합니다
print(f"MODE={MODE}  device={DEVICE}  dtype={next(model.parameters()).dtype}  "
      f"layers={CFG.num_hidden_layers}  layer_types={list(CFG.layer_types)}")

## 섹션 1. 두 메모리의 크기 (식 1)

$$|KV| = 2 \cdot L_a \cdot H_{kv} \cdot d \cdot T, \qquad |S| = L_g \cdot H \cdot d_k \cdot d_v$$

$L_a$는 full attention 레이어 수, $L_g$는 Gated DeltaNet 레이어 수, $T$는 컨텍스트 길이입니다. KV cache는 $T$에 비례해 커지고 recurrent state는 고정 크기입니다. 두 곡선이 만나는 길이 $T^*$보다 긴 컨텍스트에서는 KV가, 짧으면 state가 메모리를 더 차지합니다.

설정 필드 이름은 설치된 `transformers/models/qwen3_5/configuration_qwen3_5.py`에서 확인한 것입니다. `layer_types`, `num_key_value_heads`, `head_dim`, `linear_num_value_heads`, `linear_key_head_dim`, `linear_value_head_dim`입니다. transformers의 state 텐서 모양은 `[B, linear_num_value_heads, d_k, d_v]`이므로 $H$에는 value head 수를 씁니다. conv1d state(kernel_size × channels)는 작아서 식에서 뺍니다.

In [ ]:
layer_types = list(CFG.layer_types)
L_a = layer_types.count("full_attention")
L_g = layer_types.count("linear_attention")
H_kv, d = CFG.num_key_value_heads, CFG.head_dim
H, d_k, d_v = CFG.linear_num_value_heads, CFG.linear_key_head_dim, CFG.linear_value_head_dim
BYTES_16 = 2


def kv_elems(T):
    return 2 * L_a * H_kv * d * T


S_elems = L_g * H * d_k * d_v
T_star = S_elems / (2 * L_a * H_kv * d)

print(f"L_a={L_a} L_g={L_g} | H_kv={H_kv} d={d} | H={H} d_k={d_k} d_v={d_v}")
print(f"|KV| per token = {kv_elems(1):,} elements ({kv_elems(1) * BYTES_16 / 1024:.1f} KB at 16 bit)")
print(f"|S|            = {S_elems:,} elements ({S_elems * BYTES_16 / 1e6:.2f} MB at 16 bit)")
print(f"crossover T*   = {T_star:,.0f} tokens")
RESULTS["eq1_T_star_tokens"] = round(T_star)

In [ ]:
Ts = np.logspace(np.log10(1024), np.log10(262144), 60)
kv_gb = np.array([kv_elems(T) * BYTES_16 / 1e9 for T in Ts])
s_gb = np.full_like(kv_gb, S_elems * BYTES_16 / 1e9)

plt.figure()
plt.plot(Ts, kv_gb, label=r"$|KV|(T)$, 16 bit")
plt.plot(Ts, s_gb, label=r"$|S|$, 16 bit")
plt.axvline(T_star, ls="--", color="gray", lw=1)
plt.annotate(f"$T^*$={T_star:,.0f}", (T_star, s_gb[0]), textcoords="offset points", xytext=(6, 8))
plt.xscale("log"); plt.yscale("log")
plt.xlabel("context length T (tokens)"); plt.ylabel("memory (GB)")
plt.title(f"eq. 1, {'tiny' if MODE == 'tiny' else REAL_MODEL}")
plt.legend(); save_fig("eq1_memory_vs_T"); plt.show()

# 실제 DynamicCache 로 확인합니다: prefill 뒤 저장된 원소 수가 식과 같아야 합니다
T0 = PREFILL
ids = sample_ids(T0).to(DEVICE)
with torch.no_grad():
    cache = make_cache(model)
    model(input_ids=ids, past_key_values=cache, use_cache=True)
kv_measured = sum(l.keys.numel() + l.values.numel() for i, l in enumerate(cache.layers) if layer_types[i] == "full_attention")
s_measured = sum(l.recurrent_states[0].numel() for i, l in enumerate(cache.layers) if layer_types[i] == "linear_attention")
print(f"T0={T0}: KV measured {kv_measured:,} vs formula {kv_elems(T0):,} | S measured {s_measured:,} vs formula {S_elems:,}")
assert kv_measured == kv_elems(T0) and s_measured == S_elems
RESULTS["eq1_cache_count_match"] = "PASS"
print("PASS: DynamicCache element counts match eq. 1")

## 섹션 2. Gated delta rule (식 2)

transformers의 `torch_recurrent_gated_delta_rule`(설치된 `modeling_qwen3_5.py`)은 매 토큰마다 다음 순서로 계산합니다. 먼저 state에 decay를 곱하고, 그 다음 delta 보정을 더합니다.

$$S_t = \alpha_t S_{t-1} + \beta_t\, k_t \left(v_t - \alpha_t S_{t-1}^{\top} k_t\right)^{\top}
      = \alpha_t \left(I - \beta_t k_t k_t^{\top}\right) S_{t-1} + \beta_t k_t v_t^{\top}, \qquad o_t = S_t^{\top} q_t$$

여기서 $\alpha_t = \exp(g_t) \in (0,1)$, $\beta_t = \sigma(b_t) \in (0,1)$, $S \in \mathbb{R}^{d_k \times d_v}$입니다. 보정항 안의 $S_{t-1}$에 $\alpha_t$가 붙는다는 점이 중요합니다. 이것이 빠진 형태 $S_t = \alpha_t S_{t-1} + \beta_t k_t (v_t - S_{t-1}^{\top} k_t)^{\top}$는 다른 점화식이며, 아래 검증 셀에서 두 형태의 오차를 함께 출력해 구현과 일치하는 쪽을 확인합니다.

커널은 recurrence 전에 $q, k$를 fla 방식 l2norm(eps $10^{-6}$)으로 정규화하고 $q$를 $\sqrt{d_k}$로 나눕니다. reference 구현도 같은 전처리를 거칩니다. 이 셀이 통과해야 섹션 4에서 쓰는 전이 행렬 $M_j = \alpha_j (I - \beta_j k_j k_j^{\top})$ 해석이 transformers 구현과 같다는 근거가 됩니다.

In [ ]:
def preprocess_qk(q, k, l2norm=True):
    """transformers 가 recurrence 전에 하는 전처리: fla 식 l2norm(eps 1e-6), q / sqrt(d_k)"""
    if l2norm:
        q = q * torch.rsqrt((q * q).sum(-1, keepdim=True) + 1e-6)
        k = k * torch.rsqrt((k * k).sum(-1, keepdim=True) + 1e-6)
    return q / math.sqrt(q.shape[-1]), k


def reference_gdn(q, k, v, alpha, beta, S0=None, decay_first=True):
    """단일 head. q,k: [T,d_k]  v: [T,d_v]  alpha,beta: [T].  decay_first=False 는 보정항에 decay 가 빠진 형태입니다."""
    S = torch.zeros(k.shape[1], v.shape[1], dtype=q.dtype) if S0 is None else S0.clone()
    out = torch.zeros(v.shape, dtype=q.dtype)
    for t in range(k.shape[0]):
        S_prev = alpha[t] * S if decay_first else S
        S = alpha[t] * S + beta[t] * torch.outer(k[t], v[t] - S_prev.T @ k[t])
        out[t] = S.T @ q[t]
    return out, S

In [ ]:
from transformers.models.qwen3_5.modeling_qwen3_5 import torch_recurrent_gated_delta_rule

B, T2, H2, dk2, dv2 = 1, 24, 3, 16, 16
g = torch.Generator().manual_seed(1)
q2 = torch.randn(B, T2, H2, dk2, generator=g); k2 = torch.randn(B, T2, H2, dk2, generator=g)
v2 = torch.randn(B, T2, H2, dv2, generator=g)
log_alpha2 = -F.softplus(torch.randn(B, T2, H2, generator=g))     # g <= 0, alpha = exp(g) in (0,1)
beta2 = torch.sigmoid(torch.randn(B, T2, H2, generator=g))

out_tf, S_tf = torch_recurrent_gated_delta_rule(q2, k2, v2, g=log_alpha2, beta=beta2, initial_state=None,
                                                output_final_state=True, use_qk_l2norm_in_kernel=True)
print("transformers output", tuple(out_tf.shape), "final state", tuple(S_tf.shape))

err_exact, err_naive = 0.0, 0.0
for h in range(H2):
    qh, kh = preprocess_qk(q2[0, :, h], k2[0, :, h])
    o_ref, S_ref = reference_gdn(qh, kh, v2[0, :, h], log_alpha2[0, :, h].exp(), beta2[0, :, h])
    err_exact = max(err_exact, (o_ref - out_tf[0, :, h]).abs().max().item(), (S_ref - S_tf[0, h]).abs().max().item())
    o_nv, _ = reference_gdn(qh, kh, v2[0, :, h], log_alpha2[0, :, h].exp(), beta2[0, :, h], decay_first=False)
    err_naive = max(err_naive, (o_nv - out_tf[0, :, h]).abs().max().item())

print(f"max |reference - transformers|  (decay-first form) : {err_exact:.2e}")
print(f"max |reference - transformers|  (no decay in corr.) : {err_naive:.2e}")
ok = err_exact < 1e-4
RESULTS["eq2_max_abs_err"] = f"{err_exact:.1e}"
RESULTS["eq2_match"] = "PASS" if ok else "FAIL"
print("PASS" if ok else "FAIL", ": reference_gdn matches torch_recurrent_gated_delta_rule (threshold 1e-4)")

## 섹션 3. Fake 양자화 연산자 (식 3)

$$Q_b(x) = \left(\mathrm{clamp}\!\left(\mathrm{round}(x/s) + z,\ 0,\ 2^b-1\right) - z\right)\cdot s, \qquad s = \frac{\max - \min}{2^b - 1}, \quad z = \mathrm{round}(-\min/s)$$

`hybridquant.quant.fake_quant`가 이 연산을 마지막 차원의 group 단위로 수행합니다(비대칭 min/max, bits $\ge 16$이면 identity). 균일 양자화의 오차는 대략 $s/\sqrt{12}$에 비례하므로, 비트가 1 늘면 $s$가 약 절반이 되어 상대 L2 오차도 약 절반이 되어야 합니다. 정확히는 $b_1 \to b_2$에서 기대 비율이 $(2^{b_1}-1)/(2^{b_2}-1)$입니다. group이 커지면 한 group의 min/max 범위가 넓어져 오차가 커집니다.

In [ ]:
def rel_l2_table(x, bits=(2, 3, 4, 8), groups=(32, 64, 128, -1)):
    rows = {}
    for b in bits:
        rows[b] = {g: ((fake_quant(x, QuantSpec(b, g)) - x).norm() / x.norm()).item() for g in groups}
    return pd.DataFrame(rows).T.rename_axis("bits").rename(columns=lambda g: f"group={g}")


x_gauss = torch.randn(256, 512)
tab = rel_l2_table(x_gauss)
print("relative L2 error, Gaussian N(0,1) [256, 512]")
print(tab.map(lambda v: f"{v:.4f}"))

In [ ]:
bits = list(tab.index)
col = "group=128"
ratios = []
print(f"{'b1->b2':8s} {'measured':>9s} {'expected':>9s} {'ratio':>7s}")
for b1, b2 in zip(bits[:-1], bits[1:]):
    measured = tab.loc[b2, col] / tab.loc[b1, col]
    expected = (2 ** b1 - 1) / (2 ** b2 - 1)
    ratios.append(measured / expected)
    print(f"{b1}->{b2:<5d} {measured:9.4f} {expected:9.4f} {measured / expected:7.3f}")
ok = all(0.7 < r < 1.3 for r in ratios)
RESULTS["eq3_bit_scaling"] = "PASS" if ok else "FAIL"
print("PASS" if ok else "FAIL", ": error shrinks with bits as (2^b1-1)/(2^b2-1) within 30%")
assert fake_quant(x_gauss, QuantSpec(16, 128)) is x_gauss, "bits>=16 must be identity"

### 검증 2. 실제 모델의 K, V, S 텐서

prefill 뒤 캐시에서 첫 attention 레이어의 K, V와 첫 GDN 레이어의 S를 꺼내 같은 표를 만듭니다. 세 텐서 중 어느 것이 같은 비트에서 오차가 가장 큰지, 값 분포에 outlier가 있는지(log y축 히스토그램) 봅니다. `tiny` 모드에서는 랜덤 가중치라 수치에 의미가 없고 코드가 도는지만 확인합니다.

In [ ]:
with torch.no_grad():
    cache = make_cache(model)
    model(input_ids=sample_ids(PREFILL).to(DEVICE), past_key_values=cache, use_cache=True)
a_idx = layer_types.index("full_attention"); g_idx = layer_types.index("linear_attention")
tensors = {
    "K": cache.layers[a_idx].keys[0].float().cpu(),                 # [H_kv, T, d]
    "V": cache.layers[a_idx].values[0].float().cpu(),
    "S": cache.layers[g_idx].recurrent_states[0][0].float().cpu(),  # [H, d_k, d_v]
}
rows = {}
for name, t in tensors.items():
    rows[name] = {b: ((fake_quant(t, QuantSpec(b, 128)) - t).norm() / t.norm()).item() for b in (2, 4, 8)}
real_tab = pd.DataFrame(rows).T.rename(columns=lambda b: f"int{b} g128")
print(f"layer K/V={a_idx}, S={g_idx}, shapes:", {k: tuple(v.shape) for k, v in tensors.items()})
print(real_tab.map(lambda v: f"{v:.4f}"))
worst = real_tab.idxmax()
print("largest error per bit-width:", dict(worst))
RESULTS["eq3_worst_tensor_int4"] = worst["int4 g128"]
if MODE == "tiny":
    print("(tiny: random weights, numbers are not meaningful)")

fig, axes = plt.subplots(1, 3, figsize=(9, 2.8))
for ax, (name, t) in zip(axes, tensors.items()):
    ax.hist(t.flatten().numpy(), bins=100, log=True)
    ax.set_title(f"{name}  (std {t.std():.3g}, max|x| {t.abs().max():.3g})")
save_fig("eq3_kvs_histograms"); plt.show()

## 섹션 4. 오차 전파 (식 4)

KV 오차는 토큰마다 독립입니다. 각 토큰의 K/V는 저장될 때 한 번만 양자화되고 다시 쓰이지 않습니다. state는 다릅니다. 저장 시점 양자화 $\tilde S_i = Q(\hat S_i)$의 오차 $\delta_i = Q(\hat S_i) - \hat S_i$가 다음 스텝의 입력이 되어 recurrence를 타고 흐릅니다. 식 2가 $S$에 대해 선형이므로($k, v, \alpha, \beta$는 입력에서 오고 $S$에 의존하지 않습니다) 깨끗한 궤적 $S^*_t$와 양자화 궤적 $\tilde S_t$의 차이는 정확히 다음과 같습니다.

$$\tilde S_t - S^*_t = \sum_{i \le t} \left(\prod_{j=i+1}^{t} M_j\right) \delta_i, \qquad M_j = \alpha_j \left(I - \beta_j k_j k_j^{\top}\right)$$

$\|k_j\|_2 = 1$(l2norm)이고 $\beta_j \in (0,1)$이면 $I - \beta_j k_j k_j^{\top}$의 고유값은 1과 $1-\beta_j$이므로 $\|M_j\|_2 = \alpha_j$입니다. 따라서 $i$번째 오차가 $t$까지 살아남는 비율의 상한은 $\prod_{j=i+1}^{t} \alpha_j$입니다.

- 코드 A: 합성 입력에서 좌변(직접 계산)과 우변(명시적 곱)을 비교합니다. 둘의 차이가 부동소수점 오차 수준이면 식 4가 정확히 성립합니다.
- 코드 B: 실제 모델에서 레이어별 $\alpha$ 분포를 모아 "오차가 절반으로 줄어드는 토큰 수"를 계산합니다. $\alpha^n = 0.5$이므로 $n = \ln 0.5 / \overline{\ln \alpha} = \ln 0.5 / \bar g$입니다.
- 코드 C: `decode_nll`로 네 설정의 위치별 NLL을 실측합니다.

In [ ]:
def gdn_trajectory(q, k, v, alpha, beta, spec=None):
    """decay-first 점화식. spec 이 있으면 매 스텝 저장 전에 state 를 fake_quant 합니다.
    저장된 state 목록과 delta_t = Q(S^_t) - S^_t 목록을 돌려줍니다."""
    S = torch.zeros(k.shape[1], v.shape[1], dtype=q.dtype)
    states, deltas = [], []
    for t in range(k.shape[0]):
        S = alpha[t] * S
        S_hat = S + beta[t] * torch.outer(k[t], v[t] - S.T @ k[t])
        S = fake_quant(S_hat, spec) if spec is not None else S_hat
        states.append(S.clone()); deltas.append(S - S_hat)
    return states, deltas


def propagated_error(k, alpha, beta, deltas, t):
    """식 4 우변: sum_i (prod_{j=i+1..t} M_j) delta_i"""
    I = torch.eye(k.shape[1], dtype=k.dtype)
    M = [alpha[j] * (I - beta[j] * torch.outer(k[j], k[j])) for j in range(t + 1)]
    e = torch.zeros_like(deltas[0])
    for i in range(t + 1):
        term = deltas[i]
        for j in range(i + 1, t + 1):
            term = M[j] @ term
        e = e + term
    return e

In [ ]:
# 코드 A: 합성 검증 (float64)
T4, dk4, dv4 = 24, 16, 16
g = torch.Generator().manual_seed(0)
q4 = torch.randn(T4, dk4, generator=g, dtype=torch.float64); k4 = torch.randn(T4, dk4, generator=g, dtype=torch.float64)
v4 = torch.randn(T4, dv4, generator=g, dtype=torch.float64)
alpha4 = (-F.softplus(torch.randn(T4, generator=g, dtype=torch.float64))).exp()
beta4 = torch.sigmoid(torch.randn(T4, generator=g, dtype=torch.float64))
q4, k4 = preprocess_qk(q4, k4)

clean, _ = gdn_trajectory(q4, k4, v4, alpha4, beta4)
worst_rel = 0.0
for bits in (2, 4):
    noisy, deltas = gdn_trajectory(q4, k4, v4, alpha4, beta4, QuantSpec(bits, -1))
    for t in (0, 5, T4 - 1):
        direct = noisy[t] - clean[t]
        formula = propagated_error(k4, alpha4, beta4, deltas, t)
        rel = ((direct - formula).abs().max() / direct.abs().max()).item()
        worst_rel = max(worst_rel, rel)
        print(f"int{bits} t={t:2d}: max|lhs|={direct.abs().max():.3e}  max|lhs-rhs|={(direct - formula).abs().max():.1e}  rel={rel:.1e}")
ok = worst_rel < 1e-10
RESULTS["eq4A_identity_rel_err"] = f"{worst_rel:.1e}"
RESULTS["eq4A_identity"] = "PASS" if ok else "FAIL"
print("PASS" if ok else "FAIL", ": eq. 4 holds to float64 round-off")

# 보조 확인: ||M_j||_2 == alpha_j (||k||=1)
norms = [torch.linalg.matrix_norm(alpha4[j] * (torch.eye(dk4, dtype=torch.float64) - beta4[j] * torch.outer(k4[j], k4[j])), ord=2).item() for j in range(T4)]
print(f"max |  ||M_j||_2 - alpha_j  | = {max(abs(n - a) for n, a in zip(norms, alpha4.tolist())):.1e}")

### 코드 B. 실제 모델의 레이어별 decay 통계

GDN 모듈은 `g = -exp(A_log) * softplus(a + dt_bias)`, `beta = sigmoid(b)`로 계산합니다(`a`, `b`는 `in_proj_a`, `in_proj_b`의 출력, 설치된 `modeling_qwen3_5.py` 기준). 두 Linear에 forward hook을 걸어 `a`, `b`를 모으고 같은 식으로 $g$, $\beta$를 재계산합니다. 레이어별로 $\alpha = e^{g}$의 평균과 분위수, 그리고 반감 토큰 수 $n_{1/2} = \ln 0.5 / \bar g$를 냅니다. $n_{1/2}$가 큰 레이어일수록 state 양자화 오차가 오래 살아남습니다.

In [ ]:
captured = {}
hooks = []
for i, layer in enumerate(model.model.layers):
    if layer_types[i] != "linear_attention":
        continue
    m = layer.linear_attn
    hooks.append(m.in_proj_a.register_forward_hook(lambda mod, inp, out, i=i: captured.setdefault(i, {}).__setitem__("a", out.detach())))
    hooks.append(m.in_proj_b.register_forward_hook(lambda mod, inp, out, i=i: captured.setdefault(i, {}).__setitem__("b", out.detach())))
with torch.no_grad():
    model(input_ids=sample_ids(PREFILL + DECODE).to(DEVICE), use_cache=False)
for h in hooks:
    h.remove()

rows = []
for i in sorted(captured):
    m = model.model.layers[i].linear_attn
    g_val = (-m.A_log.float().exp() * F.softplus(captured[i]["a"].float() + m.dt_bias.float()))[0]   # [T, H]
    beta_val = torch.sigmoid(captured[i]["b"].float())[0]
    alpha_val = g_val.exp()
    q10, q50, q90 = torch.quantile(alpha_val.flatten().cpu(), torch.tensor([0.1, 0.5, 0.9])).tolist()
    rows.append({"layer": i, "alpha_mean": alpha_val.mean().item(), "alpha_p10": q10, "alpha_p50": q50, "alpha_p90": q90,
                 "beta_mean": beta_val.mean().item(), "half_life_tokens": math.log(0.5) / g_val.mean().item()})
decay_tab = pd.DataFrame(rows).set_index("layer")
print(decay_tab.round(4))
RESULTS["eq4B_half_life_min_max"] = f"{decay_tab.half_life_tokens.min():.2f} / {decay_tab.half_life_tokens.max():.2f}"
if MODE == "tiny":
    print("(tiny: random weights, numbers are not meaningful)")

plt.figure()
plt.bar(decay_tab.index.astype(str), decay_tab.half_life_tokens)
plt.yscale("log"); plt.xlabel("GDN layer index"); plt.ylabel(r"$n_{1/2}$ (tokens), $\alpha^n = 0.5$")
plt.title("state error half-life per layer (upper bound via $\\|M_j\\|_2 = \\alpha_j$)")
save_fig("eq4_half_life_per_layer"); plt.show()

### 코드 C. 실측: decode 위치별 NLL

`decode_nll`을 (KV bits, state bits) = (16,16), (16,4), (4,16), (4,4)로 돌려 위치별 NLL의 이동평균을 그립니다. state 오차가 decode 중에 쌓인다면 (16,4) 곡선이 뒤로 갈수록 (16,16)에서 멀어져야 합니다. `real` 모드에서만 의미가 있고 `tiny`에서는 실행만 확인합니다.

In [ ]:
ids_c = sample_ids(PREFILL + DECODE)
window = 2 if MODE == "tiny" else 16
plt.figure()
for kv, st in [(16, 16), (16, 4), (4, 16), (4, 4)]:
    r = decode_nll(model, ids_c, prefill_len=PREFILL, cfg=HybridQuantConfig.uniform(kv, st), max_decode=DECODE)
    curve = np.convolve(r.nll_per_pos, np.ones(window) / window, mode="valid")
    plt.plot(np.arange(len(curve)) + window, curve, label=f"{r.label}  mean={r.mean_nll:.3f}")
    print(f"{r.label:28s} mean NLL {r.mean_nll:.4f} | {r.stats}")
    RESULTS[f"eq4C_nll_kv{kv}_s{st}"] = round(r.mean_nll, 4)
plt.xlabel("decode position"); plt.ylabel(f"NLL (moving avg {window})"); plt.legend(fontsize=7)
plt.title(f"teacher-forced decode NLL ({MODE})")
save_fig("eq4_nll_vs_position"); plt.show()

## 섹션 5. 공동 예산 (식 5)

$$\min_{b_{kv},\, b_s} \ \mathcal{L}(b_{kv}, b_s) \quad \text{s.t.} \quad b_{kv} \cdot |KV|(T) + b_s \cdot |S| \le B$$

섹션 1의 크기 함수로 $(b_{kv}, b_s)$ 조합별 총 메모리를 $T = 4\text{k}, 32\text{k}, 128\text{k}$에서 계산합니다. `results/*.jsonl`이 있으면 같은 조합의 mean NLL을 표에 붙입니다. 없으면 메모리 열만 나오고 실험 뒤에 채워집니다. 등고선 그림은 $T = 32\text{k}$ 기준 총 메모리이고, NLL 격자가 완전하면 겹쳐 그립니다.

In [ ]:
BITS = [2, 4, 8, 16]
T_LIST = {"4k": 4096, "32k": 32768, "128k": 131072}


def total_gb(b_kv, b_s, T):
    return (b_kv * kv_elems(T) + b_s * S_elems) / 8 / 1e9


rows = []
for b_kv in BITS:
    for b_s in BITS:
        rows.append({"b_kv": b_kv, "b_s": b_s, **{f"GB@{n}": total_gb(b_kv, b_s, T) for n, T in T_LIST.items()}})
budget = pd.DataFrame(rows)

nll = {}
jsonl_files = sorted(glob.glob(str(ROOT / "results" / "*.jsonl")))
for f in jsonl_files:
    for line in open(f):
        r = json.loads(line)
        nll[(r["kv_bits"], r["state_bits"])] = (r["mean_nll"], r.get("model", "?"), Path(f).name)
if nll:
    budget["mean_nll"] = [nll.get((a, b), (np.nan,))[0] for a, b in zip(budget.b_kv, budget.b_s)]
    budget["source"] = [nll.get((a, b), (None, None, ""))[2] for a, b in zip(budget.b_kv, budget.b_s)]
    print(f"NLL from {len(jsonl_files)} file(s); models: {sorted({v[1] for v in nll.values()})}")
    if MODE == "tiny":
        print("(note: NLL columns come from stored real-model runs, memory columns from the tiny config)")
else:
    print("results/*.jsonl not found: memory columns only, NLL column will be filled after the grid run")
print(budget.round(4).to_string(index=False))
RESULTS["eq5_nll_cells_available"] = len(nll)

In [ ]:
Bkv, Bs = np.meshgrid(BITS, BITS, indexing="ij")
Z = np.vectorize(lambda a, b: total_gb(a, b, T_LIST["32k"]))(Bkv, Bs)

plt.figure(figsize=(5, 4))
cs = plt.contour(Bkv, Bs, Z, levels=8, cmap="viridis")
plt.clabel(cs, fmt="%.3g GB", fontsize=7)
if nll:
    N = np.array([[nll.get((a, b), (np.nan,))[0] for b in BITS] for a in BITS])
    if not np.isnan(N).any():
        cn = plt.contour(Bkv, Bs, N, levels=8, cmap="autumn", linestyles="--")
        plt.clabel(cn, fmt="NLL %.2f", fontsize=7)
    else:
        for a in BITS:
            for b in BITS:
                if not np.isnan(nll.get((a, b), (np.nan,))[0]):
                    plt.annotate(f"{nll[(a, b)][0]:.2f}", (a, b), fontsize=7, color="C3", ha="center", va="bottom")
        plt.scatter(*zip(*[k for k in nll if k[0] in BITS and k[1] in BITS]), color="C3", s=12, label="measured NLL")
        plt.legend(fontsize=7)
plt.xscale("log", base=2); plt.yscale("log", base=2)
plt.xticks(BITS, BITS); plt.yticks(BITS, BITS)
plt.xlabel(r"$b_{kv}$"); plt.ylabel(r"$b_s$"); plt.title("total memory at T=32k (solid), NLL (dashed / dots)")
save_fig("eq5_budget_contours"); plt.show()

## 검증 요약

각 섹션의 PASS/FAIL과 핵심 수치입니다. 값은 위 셀들이 실제로 출력한 것입니다.

In [ ]:
summary = pd.DataFrame({"value": RESULTS}).rename_axis("check")
print(f"MODE={MODE}")
print(summary.to_string())
fails = [k for k, v in RESULTS.items() if v == "FAIL"]
print("\nALL PASS" if not fails else f"\nFAILED: {fails}")